# Ejemplo 1 de Wenu: Primera carta celeste

Este cuaderno presenta el flujo de trabajo básico para crear una carta
astronómica con Wenu.

La carta final incluirá:

- estrellas del catálogo Hipparcos;
- líneas y etiquetas de las constelaciones occidentales;
- límites de las constelaciones definidos por la IAU;
- la eclíptica;
- el plano galáctico;
- puntos de referencia celestes;
- una retícula de coordenadas ecuatoriales.

El ejemplo mantiene separadas las partes principales del proceso:

1. definir el observador;
2. construir la escena celeste;
3. elegir una proyección;
4. dibujar la carta.

Esto facilita la comprensión de cómo Wenu construye una carta y de cómo cada
componente puede personalizarse posteriormente.


## Importaciones

Skyfield proporciona la escala de tiempo y las efemérides planetarias utilizadas
para definir el observador. Matplotlib proporciona el lienzo de dibujo.

Wenu proporciona el observador, la esfera celeste, las retículas de coordenadas,
los recursos de datos astronómicos y la proyección estereográfica.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import wenu
print(f"Wenu version: {wenu.__version__}")

from wenu.observer import Observer
from wenu.projection import StereographicProjection
from wenu.sky import CelestialSphere
from wenu.sky.coordinate_grids import EclipticGrid, GalacticGrid

## Definir la carta

El primer paso es definir el observador y la apariencia de la carta.

Los parámetros siguientes especifican el lugar de observación, la fecha y hora,
la magnitud estelar límite, la proyección estereográfica y algunas propiedades
visuales de la carta.

En cuadernos posteriores exploraremos cómo cada uno de estos parámetros afecta
el resultado final.


In [ ]:
# ---------------------------------------------------------------------
# Carta
# ---------------------------------------------------------------------

MAGNITUDE_LIMIT = 5.5

PROJECTION_RADIUS = 2.0
FLIP_EAST_WEST = True

SKY_COLOR = "slateblue"

SELECTED_CONSTELLATIONS = None

RIGHT_ASCENSIONS = range(0, 360, 30)
DECLINATIONS = (-60, -30, 0, 30, 60)

## Crear el observador

Una carta astronómica depende tanto del lugar de observación como de la hora.

Wenu almacena esta información en un objeto `Observer`. El observador también
proporciona las transformaciones de coordenadas necesarias para convertir
posiciones celestes en altura y acimut.


In [ ]:
# ---------------------------------------------------------------------
# Observador
# ---------------------------------------------------------------------

observer = Observer(
    location="La Ligua",
    time="2026-08-15 21:00",
)

In [ ]:
print("Hora UTC:", observer.t.utc_iso())
print("Latitud:", observer.lat_deg)
print("Longitud:", observer.lon_deg)
print("Elevación:", observer.elevation_m, "m")

## Crear la esfera celeste

`CelestialSphere` es el objeto central de Wenu.

Almacena los objetos astronómicos y las estructuras de referencia que aparecerán
en la carta. A medida que se agregan nuevas capas, estas pasan a formar parte de
la escena que posteriormente será proyectada y dibujada.

En este cuaderno agregaremos:

- estrellas;
- líneas de constelaciones;
- límites de constelaciones;
- puntos de referencia celestes;
- la eclíptica;
- el plano galáctico.


In [ ]:
sky = CelestialSphere(observer)

## Agregar las estrellas

Las estrellas suelen ser la primera capa astronómica que se agrega a la esfera
celeste.

Este ejemplo utiliza el catálogo Hipparcos y muestra todas las estrellas más
brillantes que la magnitud 5,5.


In [ ]:
stars = sky.add_stars(
    catalog="hipparcos",
    magnitude_limit=MAGNITUDE_LIMIT,
)

El objeto `Stars` carga el catálogo seleccionado, calcula las posiciones
aparentes de las estrellas para el observador y las prepara para ser dibujadas.

Otros catálogos pueden incorporarse a Wenu de exactamente la misma manera.


## Agregar las constelaciones

Las figuras de las constelaciones conectan estrellas seleccionadas para formar
patrones reconocibles.

Wenu mantiene las figuras de las constelaciones separadas del catálogo estelar.
Las figuras hacen referencia a identificadores del catálogo, por lo que las
estrellas deben agregarse antes que la capa de constelaciones.

Este ejemplo utiliza el sistema occidental de constelaciones.


In [ ]:
constellations = sky.add_constellations(
    system="western",
    selected=SELECTED_CONSTELLATIONS,
)

Cuando `SELECTED_CONSTELLATIONS` es `None`, Wenu incluye todas las
constelaciones disponibles en el sistema seleccionado.

En cuadernos posteriores veremos cómo dibujar solamente un grupo determinado de
constelaciones.


## Agregar los límites de las constelaciones

Los límites oficiales de la IAU dividen la esfera celeste en 88 regiones
correspondientes a las constelaciones.

Estos límites son independientes de las figuras de las constelaciones. Una
figura de constelación es un dibujo cultural, mientras que un límite define una
región oficial del cielo.


In [ ]:
boundaries = sky.add_constellation_boundaries(
    boundaries="iau",
    constellations=SELECTED_CONSTELLATIONS,
)

boundaries.sample()

## Agregar puntos de referencia celestes

Además de estrellas y constelaciones, las cartas astronómicas suelen incluir
puntos de referencia que ayudan a orientar al observador.

Wenu proporciona una colección de puntos de referencia celestes de uso habitual,
entre ellos los polos celestes, los polos eclípticos, el centro galáctico y los
puntos cardinales de la eclíptica.


In [ ]:
points = sky.add_points()

El polo celeste visible depende de la latitud del observador. Como este
ejemplo corresponde al hemisferio sur, el Polo Celeste Sur se dibujará
automáticamente.


In [ ]:
points.add_equatorial_pole(
    pole="visible",
    marker="+",
    label="SCP",
    size=120,
    color="white",
)

A continuación agregamos algunos puntos de referencia adicionales que se
muestran con frecuencia en las cartas astronómicas.


In [ ]:
points.add_ecliptic_pole(
    pole="south",
    marker="+",
    label="SEP",
    size=80,
    color="yellow",
)

points.add_galactic_center(
    marker="+",
    label="GC",
    size=80,
    color="lightblue",
)

points.add_ecliptic_keypoints(
    marker="+",
    size=70,
    color="cyan",
)

## Agregar la eclíptica

La eclíptica es la trayectoria anual aparente del Sol sobre la esfera celeste.

Wenu la construye en coordenadas eclípticas y luego la transforma al cielo
aparente del observador.


In [ ]:
ecliptic_grid = EclipticGrid(
    observer=observer,
    equinox="of_date",
)

ecliptic = ecliptic_grid.ecliptic()

El objeto `EclipticGrid` también puede utilizarse para construir meridianos
eclípticos y círculos de latitud. Aquí utilizamos solamente su curva principal:
la latitud eclíptica cero.


## Agregar el plano galáctico

El plano galáctico traza el plano central de la Vía Láctea.

Está definido por la latitud galáctica cero y se transforma al cielo aparente
del observador de la misma manera que la eclíptica.


In [ ]:
galactic_grid = GalacticGrid(
    observer=observer,
)

galactic_plane = galactic_grid.galactic_plane()

La escena astronómica está ahora completa.

Contiene estrellas, figuras de constelaciones, límites oficiales, puntos de
referencia celestes, la eclíptica y el plano galáctico.

El siguiente paso es elegir cómo se proyectará esta esfera celeste sobre el
plano de la carta.


## Elegir una proyección

La esfera celeste es una superficie tridimensional, mientras que una carta es
un dibujo bidimensional.

Una proyección define cómo se representan sobre el plano de la carta las
posiciones de la esfera celeste.

En este ejemplo utilizamos una proyección estereográfica centrada en el cenit.
Esta proyección conserva los ángulos y representa cada círculo máximo como un
círculo o una línea recta, por lo que resulta especialmente apropiada para las
cartas astronómicas.


In [ ]:
projection = StereographicProjection(
    radius=PROJECTION_RADIUS,
    flip_ew=FLIP_EAST_WEST,
)

## Crear el lienzo de dibujo

Wenu se encarga de los cálculos astronómicos y de la representación de la escena
celeste. En este cuaderno utilizamos Matplotlib para proporcionar el lienzo sobre
el cual se mostrará la carta.

El cielo visible está representado por un horizonte circular.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

fig.patch.set_alpha(0)

ax.set_aspect("equal")
ax.axis("off")

ax.set_xlim(
    -1.05 * PROJECTION_RADIUS,
     1.05 * PROJECTION_RADIUS,
)

ax.set_ylim(
    -1.05 * PROJECTION_RADIUS,
     1.05 * PROJECTION_RADIUS,
)

horizon = Circle(
    (0, 0),
    PROJECTION_RADIUS,
    facecolor=SKY_COLOR,
    edgecolor="black",
    linewidth=1.5,
)

_ = ax.add_patch(horizon)
plt.close(fig) # No dibujar nada todavía...

## Dibujar la esfera celeste

Todo está ahora preparado.

La esfera celeste contiene los objetos astronómicos, la proyección los transforma
al plano y Matplotlib proporciona la superficie de dibujo.

Para representar la carta basta con pedir a cada capa que se dibuje utilizando
la proyección elegida.


In [ ]:
# Estrellas, líneas, etiquetas y límites de las constelaciones
sky.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
sky.draw_equatorial_grid(
    ax=ax,
    projection=projection,
    ra=RIGHT_ASCENSIONS,
    dec=DECLINATIONS,
    color="white",
    linewidth=0.4,
    alpha=0.35,
)

In [ ]:
ecliptic.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="orange",
    linewidth=1.2,
)

In [ ]:
galactic_plane.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="lightblue",
    linewidth=1.2,
)

In [ ]:
points.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
ax.set_title(
    "Cielo austral\n15 de agosto de 2026, 21:00 Chile",
    fontsize=16,
)

fig.text(
    0.99,
    0.01,
    f"Generated with Wenu {wenu.__version__}",
    ha="right",
    va="bottom",
    fontsize=6,
    alpha=0.7,
)
display(fig)

## Guardar la carta

La carta terminada puede exportarse como una imagen ráster de alta resolución o
como un gráfico vectorial para publicación.


In [ ]:
output_file = Path("primera_carta_celeste.png")

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
)

print(f"Guardado en: {output_file.resolve()}")